In [1]:
%%writefile sebai.py

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class SebConfig:
    vocab_size = 256
    context_length = 128
    embedding_dim = 256
    num_heads = 8
    num_layers = 4
    dropout = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.embedding_dim % config.num_heads == 0

        self.num_heads = config.num_heads
        self.head_dim = config.embedding_dim // config.num_heads

        self.qkv = nn.Linear(config.embedding_dim, config.embedding_dim * 3)
        self.proj = nn.Linear(config.embedding_dim, config.embedding_dim)

        self.register_buffer(
            "mask",
            torch.tril(
                torch.ones(
                    config.context_length,
                    config.context_length
                )
            ).view(
                1,
                1,
                config.context_length,
                config.context_length
            )
        )

    def forward(self, x):
        batch, seq, channels = x.shape

        q, k, v = self.qkv(x).chunk(3, dim=-1)

        q = q.view(batch, seq, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch, seq, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch, seq, self.num_heads, self.head_dim).transpose(1, 2)

        attention = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attention = attention.masked_fill(
            self.mask[:, :, :seq, :seq] == 0,
            float("-inf")
        )

        attention = F.softmax(attention, dim=-1)

        output = attention @ v
        output = output.transpose(1, 2).contiguous().view(batch, seq, channels)

        return self.proj(output)

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.embedding_dim * 4),
            nn.GELU(),
            nn.Linear(config.embedding_dim * 4, config.embedding_dim)
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.norm1 = nn.LayerNorm(config.embedding_dim)
        self.attention = CausalSelfAttention(config)

        self.norm2 = nn.LayerNorm(config.embedding_dim)
        self.feed_forward = FeedForward(config)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feed_forward(self.norm2(x))
        return x

class SebAI(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.embedding_dim
        )

        self.position_embedding = nn.Embedding(
            config.context_length,
            config.embedding_dim
        )

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(config)
                for _ in range(config.num_layers)
            ]
        )

        self.norm = nn.LayerNorm(config.embedding_dim)

        self.lm_head = nn.Linear(
            config.embedding_dim,
            config.vocab_size,
            bias=False
        )

        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids, targets=None):
        batch, seq = input_ids.shape

        positions = torch.arange(
            seq,
            device=input_ids.device
        )

        x = self.token_embedding(input_ids)
        x = x + self.position_embedding(positions)

        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss

config = SebConfig()
model = SebAI(config)

parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(f"Parameters: {parameters:,}")

Writing sebai.py


In [3]:
!python sebai.py

Parameters: 3,257,856


In [8]:
import torch

text = """
SebAI is a language model built from scratch.
SebAI learns by predicting the next character.
This model starts with random weights.
It is trained using PyTorch on Google Colab.
SebAI is open source software.
The model learns patterns from text.
Machine learning is the process of learning patterns from data.
A transformer uses attention to process sequences.
The goal of SebAI is to build an independent language model.
""" * 500

chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

data = torch.tensor(
    [stoi[ch] for ch in text],
    dtype=torch.long
)

print("Characters:", len(chars))
print("Tokens:", len(data))

Characters: 33
Tokens: 211000


In [10]:
from sebai import SebAI, config

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SebAI(config).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

batch_size = 32
steps = 1000

print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Parameters: 3,257,856
Device: cuda
Parameters: 3257856


In [12]:
model.train()

for step in range(1000):
    starts = torch.randint(
        0,
        len(data) - config.context_length - 1,
        (32,)
    )

    x = torch.stack([
        data[i:i + config.context_length]
        for i in starts
    ]).to(device)

    y = torch.stack([
        data[i + 1:i + config.context_length + 1]
        for i in starts
    ]).to(device)

    optimizer.zero_grad()

    logits, loss = model(x, y)

    loss.backward()

    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}: loss {loss.item():.4f}")

Step 0: loss 170.4216
Step 100: loss 2.5374
Step 200: loss 2.0060
Step 300: loss 1.9061
Step 400: loss 1.7674
Step 500: loss 1.5312
Step 600: loss 0.9355
Step 700: loss 0.4026
Step 800: loss 0.1269
Step 900: loss 0.0580


In [15]:
import torch
import torch.nn.functional as F

model.eval()

context = torch.tensor(
    [[stoi["S"]]],
    dtype=torch.long,
    device=device
)

for _ in range(300):
    idx = context[:, -config.context_length:]

    with torch.no_grad():
        logits, _ = model(idx)

    probabilities = F.softmax(logits[:, -1, :], dim=-1)

    next_token = torch.multinomial(
        probabilities,
        num_samples=1
    )

    context = torch.cat(
        [context, next_token],
        dim=1
    )

generated = "".join(
    itos[int(i)]
    for i in context[0]
)

print(generated)

SebAI is to build an independent language model.

SebAI is a language model built from scratch.
SebAI learns by predicting the next characteracter.
This momodel s statatarts wits random weights.
It is trained using PyTorch on Google Colab.
SebAI is open source software.
The model learns patterns from
